# Sorigul large-v3 — 사용자 직접 실행
먼저 별도 LIVE 승인을 받으세요. 무료 계정의 무료 GPU Runtime을 직접 선택하고 셀을 순서대로 실행합니다. 유료 Compute Unit을 사용하는 세션이면 중단하세요. GPU가 없으면 중단합니다. 서버·터널·Drive mount·백그라운드 제어는 사용하지 않습니다. 승인된 자체 제작 MP3 1개만 업로드합니다.
준비물: 이 Notebook, 같은 검토된 commit의 sorigul_notebook_code.zip, 승인된 MP3와 SHA-256. GitHub 인증정보나 Google Token을 입력하지 마세요.

In [ ]:
FREE_RUNTIME_CONFIRMED = input('유료 플랜/Compute Unit을 사용하지 않는 무료 GPU Runtime임을 확인했으면 FREE 입력: ') == 'FREE'
assert FREE_RUNTIME_CONFIRMED, '무료 Runtime 조건을 확인한 뒤 진행하세요.'
import torch
assert torch.cuda.is_available(), '무료 GPU 없음: 유료로 전환하지 말고 중단하세요.'
print('GPU:', torch.cuda.get_device_name(0), 'Torch:', torch.__version__)
%pip install -q faster-whisper==1.2.1 pydantic==2.13.4 cryptography==50.0.1 mutagen==1.48.1


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, io, json, sys, zipfile
uploaded_code = files.upload()
assert set(uploaded_code) == {'sorigul_notebook_code.zip'}, '소스 ZIP 하나만 선택하세요.'
CODE_FILES = {'colab/sorigul_colab_bootstrap.py', 'colab/sorigul_notebook.py', 'backend/src/domain/transcription.py', 'backend/src/services/audio_metadata.py', 'backend/src/services/output_bundle.py', 'backend/src/services/notebook_bundle.py'}
code_root = Path('/content/sorigul_notebook_source')
code_root.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(uploaded_code['sorigul_notebook_code.zip'])) as package:
    assert len(package.infolist()) == 7 and set(package.namelist()) == CODE_FILES | {'code_manifest.json'}
    assert sum(item.file_size for item in package.infolist()) < 1048576
    code_manifest = json.loads(package.read('code_manifest.json'))
    for name in CODE_FILES:
        content = package.read(name)
        assert hashlib.sha256(content).hexdigest() == code_manifest['sha256'][name]
        destination = code_root / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
sys.path.insert(0, str(code_root))
print('Source commit:', code_manifest['source_commit'])


In [ ]:
from faster_whisper import WhisperModel
from colab.sorigul_notebook import load_model, transcribe_once
model, identity = load_model(torch, WhisperModel)
print('Model:', identity.model, 'Actual device:', model.model.device, 'Actual compute_type:', model.model.compute_type)


In [ ]:
EXPECTED_AUDIO_SHA256 = input('승인된 자체 제작 MP3의 SHA-256: ').strip().lower()
assert len(EXPECTED_AUDIO_SHA256) == 64
uploaded_audio = files.upload()
assert len(uploaded_audio) == 1, '승인된 MP3 하나만 선택하세요.'
name, audio_bytes = next(iter(uploaded_audio.items()))
assert Path(name).name == name and Path(name).suffix.lower() == '.mp3'
assert 0 < len(audio_bytes) <= 64 * 1024 * 1024
assert hashlib.sha256(audio_bytes).hexdigest() == EXPECTED_AUDIO_SHA256, '승인된 MP3와 다릅니다. 중단하세요.'
source = code_root / name
with source.open('xb') as handle:
    handle.write(audio_bytes)


In [ ]:
assert 'SORIGUL_INFERENCE_STARTED' not in globals(), '이 승인으로 추론은 1회만 실행합니다.'
SORIGUL_INFERENCE_STARTED = True
result_zip = code_root / 'sorigul_notebook_result.zip'
payload, runtime = transcribe_once(source, model, identity, result_zip)
print(json.dumps(runtime, ensure_ascii=False))
print('전사:', payload['text'])
for segment in payload['segments']:
    print(segment['start'], segment['end'], segment['text'])
print('ZIP: TXT/JSON/SRT + 원본 연결/체크섬 manifest. 생성 결과를 기준 문장과 비교하세요.')


In [ ]:
files.download(str(result_zip))
print('Sorigul 폴더 화면에서 원본 MP3 선택 → Notebook 결과 ZIP 가져오기. Drive 전송은 별도 버튼입니다.')


검증 후 Runtime 연결을 해제합니다. 결과 ZIP은 TXT/JSON/SRT와 manifest만 포함하며 MP3는 포함하지 않습니다. Notebook 성공은 데스크톱 자동 전사 연결 성공이 아닙니다. 결과는 사용자가 직접 다운로드하고 가져옵니다.